In [ ]:
import sys, json, os
sys.path.insert(0, "../src")
from pathlib import Path
from dotenv import load_dotenv
load_dotenv("../.env")

cases = json.loads(Path("../data/eval_cases.json").read_text())
results = json.loads(Path("../data/run_results.json").read_text())

case_map = {c["id"]: c for c in cases}
print(f"Loaded {len(cases)} cases, {len(results)} run results")

In [ ]:
import pandas as pd
from arxiv_agent.eval import (
    score_tool_called,
    score_correct_paper,
    score_keyword_coverage,
    score_no_hallucination,
)

rows = []
for run in results:
    case = case_map[run["id"]]
    row = {
        "id": run["id"],
        "category": case["category"],
        "question": case["question"][:60],
        "trace_id": run["trace_id"],
        "tool_called": score_tool_called(run, case["expects_tool_call"]),
        "correct_paper": score_correct_paper(run, case.get("expected_paper_id")),
        "keyword_coverage": score_keyword_coverage(run, case.get("expected_keywords", [])),
        "no_hallucination": score_no_hallucination(run, case["expects_tool_call"]),
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(df[["id", "category", "tool_called", "correct_paper", "keyword_coverage", "no_hallucination"]])

In [ ]:
from arxiv_agent.observability import get_client, flush

lf = get_client()
score_cols = ["tool_called", "correct_paper", "keyword_coverage", "no_hallucination"]

for _, row in df.iterrows():
    for score_name in score_cols:
        val = row[score_name]
        if val is None:
            continue
        lf.score(
            trace_id=row["trace_id"],
            name=score_name,
            value=float(val),
        )

flush()
print("Scores posted to Langfuse. Open the UI to see them on each trace.")
print(f"UI: {os.environ['LANGFUSE_BASE_URL']}")

In [ ]:
print("=== Overall Scores ===")
for col in score_cols:
    valid = df[col].dropna()
    print(f"  {col}: {valid.mean():.2f} ({valid.sum():.0f}/{len(valid)} pass)")

print("\n=== By Category ===")
for cat in df["category"].unique():
    sub = df[df["category"] == cat]
    print(f"\n  {cat} ({len(sub)} cases):")
    for col in score_cols:
        valid = sub[col].dropna()
        if len(valid) > 0:
            print(f"    {col}: {valid.mean():.2f}")

In [ ]:
base_url = os.environ["LANGFUSE_BASE_URL"]

print("=== Failing Cases ===\n")
for _, row in df.iterrows():
    failures = []
    for col in score_cols:
        if row[col] is not None and row[col] < 1.0:
            failures.append(f"{col}={row[col]:.2f}")
    if failures:
        print(f"[{row['id']}] {row['question']}")
        print(f"  Failed: {', '.join(failures)}")
        print(f"  Trace: {base_url}/trace/{row['trace_id']}")
        print()